In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.7 Where Linear Algebra Stops: Nonlinearity, Piecewise-Linear Maps, and the Residual Stream

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter VIII — The Linear Algebra of Learning",
    number="8.7",
    title="Where Linear Algebra Stops: Nonlinearity, Piecewise-Linear "
    "Maps, and the Residual Stream",
    blurb="Depth without nonlinearity is one matrix; with ReLU it is "
    "exponentially many matrices, one per activation region, each exact. "
    "The chapter closes by measuring the boundary of its own subject: "
    "where the linear story ends, where it survives piecewise, and where "
    "— in normalization layers and the residual stream — it quietly "
    "remains the right language after all.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

A course about linear algebra owes its reader an honest map of the
boundary, and every border marker here is gated. First the negative
result that motivates nonlinearity: five stacked linear layers
collapse to a single matrix (gated at $10^{-13}$) — depth buys
nothing without an activation between. Then the structure ReLU
actually creates: a small network partitions its input plane into
**54 affine regions** (enumerated by activation pattern on a fine
grid), and *within each region the network is exactly affine* — the
fitted map's residual sits at $10^{-15}$ and its gradient equals the
analytic masked-product Jacobian $W_1D_1W_2D_2W_3$ at $10^{-14}$.
Linear algebra does not stop at ReLU; it multiplies — one matrix per
region, with region counts growing with depth yet never exceeding
the Zaslavsky-style bounds {cite}`montufar2014` (gated one-sidedly,
as counting theorems are).

The modern architecture keeps two linear sanctuaries, both measured.
**LayerNorm** is a projection off the all-ones direction followed by
a normalization: outputs have exactly zero mean and unit variance
(gated $10^{-14}$), and its Jacobian *annihilates* $\mathbf{1}$
(gated $10^{-14}$) — the constant direction is dead by construction,
which is [§6.1](../06-structure/graphs-laplacian.ipynb)'s null space
wearing a normalization; RMSNorm is exactly scale-invariant. And the
**residual stream** is a vector space in the plainest sense: writing
a labeled direction into it at layer 3 makes a held-out **linear
probe** ([§8.1](learning-as-least-squares.ipynb)'s least squares on
hidden states) jump from chance to 100% at exactly that layer, its
weight vector recovering the planted direction at cosine 0.83 — not
unity, and honestly so: the stream keeps mixing after the write, and
the probe finds the direction as rotated by two further layers. The
course ends where its subject genuinely stops — and finds the
stopping line drawn in linear algebra's own ink.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Montúfar et al. {cite}`montufar2014` for region counting;
> Vaswani et al. {cite}`vaswani2017` for the LayerNorm/residual
> architecture. The probing methodology is [§8.1](learning-as-least-squares.ipynb)'s
> ridge machinery applied to hidden states.

## Theory in brief

### Depth needs nonlinearity; nonlinearity keeps linearity, piecewise

A composition of linear maps is the product of their matrices —
depth collapses. With ReLU between, the input space is partitioned by
the activation pattern $\pi(x) = (\text{sign of each preactivation})$,
and on each cell,

```{math}
:label: eq-ws-region
f(x) \;=\; W_L D_{L-1}\cdots D_1 W_1\,x + (\text{bias terms}),
\qquad D_\ell = \operatorname{diag}(\pi_\ell),
```

an **exactly affine** map whose Jacobian is the masked product. The
number of cells grows with width and depth but never beyond
arrangement bounds ($\prod_\ell \sum_{j\le d}\binom{n_\ell}{j}$ for
input dimension $d$) — expressivity is *counted*, and the counts are
gateable one-sidedly.

### The linear sanctuaries

LayerNorm subtracts the mean — a projection $P = I -
\tfrac1d\mathbf{1}\mathbf{1}^{\top}$ — then rescales to unit
variance:

```{math}
:label: eq-ws-ln
\mathrm{LN}(x) = \frac{Px}{\lVert Px\rVert/\sqrt{d}},
\qquad J_{\mathrm{LN}}\mathbf{1} = 0,
```

so the all-ones direction is annihilated exactly — inputs differing
by a constant become indistinguishable, by design. RMSNorm drops the
projection and keeps the scaling, making it exactly invariant under
$x \mapsto \alpha x$ for $\alpha > 0$. Both are almost-linear
islands: fixed projections and data-dependent scales.

### The residual stream is a vector space

With updates $h_{\ell+1} = h_\ell + f_\ell(h_\ell)$, every layer
*adds* to a shared $d$-dimensional stream — so information is stored
as **directions**, superposed, and read back by inner products. A
linear probe is [§8.1](learning-as-least-squares.ipynb)'s least
squares fitted to hidden states; where it starts succeeding is where
the information entered the stream, and its weight vector *is* the
readout direction — measurable claims, both gated below.

---
## Setup

Data only: the seeded rng. The networks — linear, ReLU, normalized — are
built in the exercises, where the boundary is the lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

from ecp import validate
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps

## Exercise 1: Depth without nonlinearity is one matrix

**Part a)** Stack five linear layers ($20 \to 20$, random weights) and
gate the collapse: applying them in sequence to a batch equals one
multiplication by the precomputed product, at $10^{-13}$ scaled —
associativity, wearing an architecture.

**Part b)** Count what collapse costs: five matrices hold
$5 \cdot 400 = 2000$ parameters but reach only the $400$-dimensional
space of single matrices — gate `matrix_rank` of the product at
$\le 20$ trivially and, more tellingly, confirm the *composition of
five random rank-20 maps is still just rank 20*: no new functions,
only new parameterizations of old ones. The entire case for
activation functions is this gate.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    gap_collapse < 1e-13,
    "five linear layers equal one matrix (associativity as architecture)",
    f"{gap_collapse:.0e} relative on a 50-sample batch — depth without "
    "nonlinearity buys parameters, not functions",
)
validate.check(
    rank_prod == D_LIN,
    "and the composed map lives in the same 400-dimensional space",
    "full rank but nothing new: the entire case for activation "
    "functions, stated as a rank computation",
)

## Exercise 2: ReLU: one matrix per region, each exact

**Part a)** Build a $2 \to 8 \to 8 \to 1$ ReLU network and enumerate
its activation regions on a $400^2$ grid over $[-2,2]^2$: each
distinct pattern of preactivation signs is one region — **54** of
them here.

**Part b)** Gate {eq}`eq-ws-region` on the largest region: fit an
affine map by least squares to 200 of its points — residual below
$10^{-13}$ (the network is *exactly* affine there, not
approximately), and the fitted gradient equals the analytic masked
product $W_1D_1W_2D_2W_3$ at $10^{-13}$. The Jacobian "at a point"
is a genuine matrix — [§8.3](linear-layer-backpropagation.ipynb)'s
backpropagation computes exactly it — and it is constant until the
input crosses a fold.

**Part c)** Draw the partition: the plane coloured by region, the
decision boundary ($f = 0$) overlaid — a mosaic of flat tiles whose
seams are where the linear story hands over to the next tile.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    res_affine < 1e-13,
    "within a region the ReLU network is exactly affine (Eq. 1)",
    f"fit residual {res_affine:.0e} over 200 points — not "
    "approximately linear: linear, until the pattern flips",
)
validate.check(
    gap_jac < 1e-13,
    "and the region's matrix is the masked weight product",
    f"fitted gradient vs W1 D1 W2 D2 W3 at {gap_jac:.0e} — "
    "backpropagation's Jacobian, recovered by regression from function "
    "values alone",
)

## Exercise 3: Counting expressivity, under its bound

**Part a)** Sweep depth 1–3 at width 8 (fresh random weights each) and
count regions on the grid. Gate the theorem one-sidedly: every count
sits below the arrangement bound
$\bigl(\sum_{j\le2}\binom{8}{j}\bigr)^{L} = 37^{L}$
{cite}`montufar2014` — counts 26, 53, 128 against 37, 1369, 50653.
(Grid enumeration can only *under*count, which is the safe direction
for a $\le$ gate and is stated.)

**Part b)** Draw count against depth on a log axis with the bound:
both grow geometrically; the widening gap is the bound's looseness,
not a failure — and the *growth itself* is the honest headline:
depth multiplies tiles, which is what "expressive" means for
piecewise-linear networks.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    under_bound and growing,
    "region counts grow with depth and never exceed the arrangement "
    "bound",
    f"{[depth_counts[L] for L in Ls]} against 37^L — the one-sided "
    "counting theorem, with grid undercounting on the safe side",
)

## Exercise 4: The linear sanctuaries: LayerNorm and RMSNorm

**Part a)** Implement {eq}`eq-ws-ln` and gate its contract on 100
random vectors ($d = 32$): output mean exactly-to-rounding zero and
variance 1, both at $10^{-13}$.

**Part b)** Gate the null direction: the analytic LayerNorm Jacobian
(projection, then normalized-scaling terms) applied to $\mathbf{1}$
gives zero at $10^{-14}$ — and confirm behaviourally:
$\mathrm{LN}(x + 17c\mathbf{1}) = \mathrm{LN}(x)$ at $10^{-14}$. The
constant direction is [§6.1](../06-structure/graphs-laplacian.ipynb)'s
Laplacian null space reappearing as an invariance every transformer
carries in every layer.

**Part c)** Gate RMSNorm's exact scale-invariance:
$\mathrm{RMS}(\alpha x) = \mathrm{RMS}(x)$ for twenty positive
$\alpha$ spanning six decades, at $10^{-14}$ — the reason RMS-normed
networks are invariant to a whole family of weight rescalings.

**Part d)** Draw LayerNorm geometrically at $d = 3$: random points,
their images all landing on the circle of radius $\sqrt{3}$ inside
the zero-sum plane — normalization as projection onto a sphere
living in a subspace.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    mean_gap < 1e-13 and var_gap < 1e-13,
    "LayerNorm outputs have zero mean and unit variance (Eq. 2)",
    f"{mean_gap:.0e} and {var_gap:.0e} over 100 vectors — the contract, "
    "at rounding",
)
validate.check(
    j_ones < 1e-14 and shift_gap < 1e-14,
    "and its Jacobian annihilates the all-ones direction, exactly as "
    "the invariance demands",
    f"J@1 at {j_ones:.0e}, behavioural shift test at {shift_gap:.0e}: "
    "6.1's null space, running inside every transformer layer",
)
validate.check(
    rms_gap < 1e-14,
    "while RMSNorm is scale-invariant across six decades",
    f"{rms_gap:.0e} for alpha in [1e-3, 1e3] — an exact homogeneity, "
    "and a whole family of weight rescalings the network cannot see",
)

## Exercise 5: The residual stream, probed

**Part a)** Build a six-layer toy residual stream
($h_{\ell+1} = h_\ell + 0.3\tanh(h_\ell W_\ell)$, $d = 32$, 400
samples) and *write* a binary label into it at layer 3 only: add
$\pm2\,v$ along a fixed unit direction $v$ according to the label.

**Part b)** Probe every layer with held-out least squares
([§8.1](learning-as-least-squares.ipynb): fit on 300 states, evaluate
on 100). Gate the jump: test accuracy at chance level (below 0.65)
for layers before the write, above 0.9 from the write layer on — the
probe *locates* where information entered the stream, which is the
entire methodology of representation analysis, here on a system whose
ground truth is planted.

**Part c)** Gate the readout direction honestly: the probe weights'
cosine against the planted $v$ is **0.83** at the write layer — high,
not unity, because two further residual updates keep mixing the
stream after the write; gate $> 0.7$ and report the decay across
layers. The stream is a vector space; it is not a static one.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
One direction was planted; real streams superpose many. Ask your
assistant to extend the toy: write THREE independent labeled
directions at different layers, then probe for each separately —
and check it against the mathematics rather than a demo: (i) each
probe's accuracy jumps at its own write layer and not before;
(ii) with orthogonal planted directions the probes do not interfere
(each cosine against its own target exceeds the others by a stated
margin); (iii) with nearly parallel planted directions
(cosine 0.95) the probes DO interfere, and the accuracy cost is a
measurement of superposition's price. The check is yours.
```

### Validation 5

In [ ]:
validate.check(
    pre_ok and post_ok,
    "the held-out probe locates the write layer exactly",
    f"accuracies {[f'{a:.2f}' for a in probe_acc]} — chance before "
    "layer 3, above 0.9 from it: information entry, found by least "
    "squares on a system with planted ground truth",
)
validate.check(
    cos_at_write > 0.7,
    "and recovers the planted direction at the honest cosine",
    f"{cos_at_write:.2f} at the write layer — high, not unity: the "
    "stream keeps mixing after the write, and the probe reads the "
    "direction as two further layers rotated it",
)

---
## Notebook summary

**The boundary, surveyed with gates.** Five linear layers collapsed
to one matrix at $10^{-14}$ — depth needs nonlinearity. ReLU
answered not by abandoning linearity but by *tiling* it: 54 regions
on the demonstration network, each **exactly** affine (residual
$10^{-15}$), each with the masked-product Jacobian backpropagation
promises ($10^{-14}$), with counts growing 26 → 53 → 128 across
depths under the $37^L$ arrangement bound, one-sidedly gated.

**The sanctuaries held.** LayerNorm delivered exact zero mean and
unit variance, annihilated the constant direction in both Jacobian
and behaviour at $10^{-14}$ ([§6.1](../06-structure/graphs-laplacian.ipynb)'s
null space, running in every transformer layer), and drew as the
sphere-in-a-subspace it is; RMSNorm was scale-invariant across six
decades.

**And the residual stream answered to least squares.** A direction
written at layer 3 made held-out probes jump from chance to 100% at
exactly layer 3, recovering the planted vector at cosine 0.83 — not
unity, because the stream kept mixing, and the honest number is the
more instructive one. Where linear algebra stops, it stops piecewise,
locally, and legibly — in its own language, which is what this
course was for.

**Methods introduced.** Linear-collapse gates, activation-region
enumeration with exact affine fits, masked-product Jacobians,
one-sided region-count gating, LayerNorm/RMSNorm contracts and
Jacobian null directions, and held-out linear probing with planted
ground truth.

## Outlook

- **The epilogue remains.** [§E.1](../epilogue/five-factorizations-one-idea.ipynb)
  closes the course by laying its five factorizations side by side —
  one idea, five costumes, forty-seven notebooks of evidence.
- **Interpretability is this notebook at scale.** Probing, activation
  patching, and dictionary learning on residual streams are
  Exercises 2 and 5 industrialized — with superposition (the
  assistant exercise's third check) as the field's central obstacle.
- **Smooth activations bend, not tile.** GELU and SiLU replace exact
  regions with soft ones; the Jacobian still rules locally, but the
  exact-affine gates become approximate — a real cost, paid for
  trainability.
- **Training dynamics stayed out of scope.** Why gradient descent
  finds these particular tiles and directions is the open frontier —
  the one place this course's tools describe the object perfectly
  and the process not yet.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()